In [1]:
import os
import numpy as np
import re
import cleantext
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import time
import warnings

warnings.filterwarnings("ignore")

# Set seed for reproducibility
seed_value = 1
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

### Data loading and cleaning

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashishpandey2062/next-word-predictor-text-generator-dataset")

print(os.listdir(path))

['next_word_predictor.txt']


In [3]:
data_path = f"{path}/{os.listdir(path)[0]}"

with open(data_path, 'r', encoding='utf-8') as file:
    text = file.read()

text

'The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.\n\nAs the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.\n\nIn the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.\n\nAs the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. It was a day filled with joy, laughter, and memories that would last a lifetime.\n\n\nT

In [4]:
clean_text = cleantext.clean(
    text,
    fix_unicode=True,
    to_ascii=True,
    lower=True,
    no_line_breaks=True,   
    no_urls=True,
    no_emails=True,
    no_phone_numbers=True, 
    no_numbers=False,
    no_punct=False,
    replace_with_url="",
    replace_with_email="",
    lang="en"
)
clean_text

'the sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. people were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. children were playing games, and laughter filled the air. as the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. families gathered for picnics, and the smell of barbecues wafted through the air. it was a perfect day for a picnic by the lake. in the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. the atmosphere was electric, and the music had everyone swaying to the beat. as the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. it was a day filled with joy, laughter, and memories that would last a lifetime. the ancient cas

There are backslashes in the preceding displayed text like: \
the city\\'s cultural scene was rich and diverse

In [5]:
num_char = 25
for match in re.finditer(r"'", clean_text):
    start = match.start()
    print(repr(clean_text[start-num_char:start+num_char]))

"the corridors. the castle's library housed an impr"
"nds. at night, the castle's windows lit up with a "
"ducks and swans. the park's paths were lined with "
"the urban storm. the city's cultural scene was ric"
" those braving the desert's scorching heat. nomadi"
" remedies from the forest's flora to treat ailment"
"e shadows, and the amazon's diverse birdlife fille"
"cess to some of the world's most breathtaking view"
"s spun with each passerby's touch, releasing bless"
"derstanding of the forest's secrets, relying on it"
"ape. at night, the amazon's bioluminescent insects"
"who depended on the ocean's bounty. they weathered"
"they relied on their land's abundant geothermal en"
"ls celebrated the country's viking history, with t"
"jokull captured the world's attention when they er"
"e descriptions of iceland's unique landscapes and "
"e knowledge of the desert's secrets guided them th"
"he sand dunes, the sahara's colors transformed int"
"ngkhar puensum, the world's highest unclimbed

Turned out the string contains just a plain apostrophe

In [6]:
# save the clean file
with open('next_word_predictor_clean.txt', 'w', encoding='utf-8') as f:
    f.write(clean_text)

### Initiate and fit the tokenizer

In [7]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([clean_text])

### Convert text to sequences

In [8]:
input_sequences = []

for sentence in clean_text.split('.'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])

### Pad sequences

In [9]:
max_len = max([len(seq) for seq in input_sequences])

padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

### Split features and the target

In [10]:
X = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

n_classes = len(tokenizer.word_index) + 2
y = tf.keras.utils.to_categorical(y, num_classes=n_classes)

### Build a model

In [11]:
def create_model(model_config, X, y, epochs, learning_rate=0.001):
    assert model_config[0]["layer"] == Embedding, "The first layer must be Embedding!"

    model = Sequential()

    recurrent_layers = [layer_cfg for layer_cfg in model_config if layer_cfg["layer"] in (LSTM, GRU)]
    last_recurrent = recurrent_layers[-1] if recurrent_layers else None

    dense_layers = [layer_cfg for layer_cfg in model_config if layer_cfg["layer"] == Dense]
    last_dense = dense_layers[-1] if dense_layers else None

    for layer_cfg in model_config:
        layer = layer_cfg["layer"]

        if layer == Embedding:
            model.add(Embedding(n_classes, layer_cfg["units"], input_length=max_len-1))

        elif layer in (LSTM, GRU):
            return_seq = (layer_cfg is not last_recurrent)
            model.add(layer(layer_cfg["units"], return_sequences=return_seq))

        elif layer == Dropout:
            model.add(Dropout(layer_cfg["rate"]))

        elif layer == Dense:
            if layer_cfg is last_dense:
                model.add(Dense(n_classes, activation='softmax'))
            else:
                model.add(Dense(layer_cfg["units"], activation='relu'))

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X, y, epochs=epochs, verbose = 1)
    
    return model

In [12]:
model_config = [
    {"layer": Embedding, "units": 32},
    {"layer": LSTM, "units": 32},
    {"layer": Dropout, "rate": 0.2},
    {"layer": LSTM, "units": 16},
    {"layer": Dropout, "rate": 0.2},
    {"layer": Dense, "units": 32},
    {"layer": Dense, "units": 16},
]

epochs = 15

model_test = create_model(model_config, X, y, epochs=epochs)
model_test.summary()

Epoch 1/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 26s 29ms/step - accuracy: 0.0476 - loss: 7.1602
Epoch 2/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 23s 28ms/step - accuracy: 0.0488 - loss: 6.7090
Epoch 3/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.0489 - loss: 6.5966
Epoch 4/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 27s 33ms/step - accuracy: 0.0522 - loss: 6.5087
Epoch 5/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.0570 - loss: 6.4357
Epoch 6/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 29s 36ms/step - accuracy: 0.0605 - loss: 6.3633
Epoch 7/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 28s 35ms/step - accuracy: 0.0640 - loss: 6.2996
Epoch 8/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 27s 33ms/step - accuracy: 0.0667 - loss: 6.2339
Epoch 9/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 28s 34ms/step - accuracy: 0.0699 - loss: 6.1353
Epoch 10/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.0738 - loss: 6.0612
Epoch 11/15
809/809 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.0755 - loss: 5.9977
Epoch 12/15
809/809 ━━━━━━━━━━

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 83, 32)         │       159,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 83, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 83, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4984)           │       164,472 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,007,882 (3.84 MB)

 Trainable params: 335,960 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 671,922 (2.56 MB)

In [13]:
def predict_next_word(model, tokenizer, sentence):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    tokenized_sentence = pad_sequences([tokenized_sentence], maxlen=max_len, padding='pre')
    position = np.argmax(model.predict(tokenized_sentence, verbose=0))
    for word, index in tokenizer.word_index.items():
        if index == position:
            return word

In [14]:
my_sentence = "the castle"

predict_next_word(model_test, tokenizer, my_sentence)

'and'

In [15]:
def predict_next_n_words(model, tokenizer, sentence, n=5):
    i = 0
    while i <=n:
        next_word = predict_next_word(model_test, tokenizer, sentence)
        sentence += " " + next_word
        i += 1
    print(sentence)

In [16]:
my_sentence = "the castle"

predict_next_n_words(model_test, tokenizer, my_sentence, 5)

the castle and the land of the land


### LSTM model

In [17]:
model_config = [
    {"layer": Embedding, "units": 32},
    {"layer": LSTM, "units": 32},
    {"layer": Dense, "units": 16},
]

epochs = 50

start = time.time()
model_lstm = create_model(model_config, X, y, epochs=epochs)
end = time.time()
elapsed = end - start
print(f"Training time: {int(elapsed // 60)} min {int(elapsed % 60)} sec")

model_lstm.summary()


Epoch 1/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - accuracy: 0.0487 - loss: 7.1635
Epoch 2/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.0497 - loss: 6.7454
Epoch 3/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.0634 - loss: 6.6177
Epoch 4/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0684 - loss: 6.4731
Epoch 5/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.0741 - loss: 6.3422
Epoch 6/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.0846 - loss: 6.2159
Epoch 7/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.0927 - loss: 6.0879
Epoch 8/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.1021 - loss: 5.9594
Epoch 9/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.1080 - loss: 5.8398
Epoch 10/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.1116 - loss: 5.7302
Epoch 11/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.1159 - loss: 5.6276
Epoch 12/50
809/809 ━━━━━━━━━━

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 83, 32)         │       159,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4984)           │       164,472 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 996,842 (3.80 MB)

 Trainable params: 332,280 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 664,562 (2.54 MB)

In [18]:
my_sentence = "the castle"

predict_next_n_words(model_lstm, tokenizer, my_sentence, 5)

the castle and the land of the land


### GRU model

In [19]:
model_config = [
    {"layer": Embedding, "units": 32},
    {"layer": GRU, "units": 32},
    {"layer": Dense, "units": 16},
]

epochs = 50

start = time.time()
model_gru = create_model(model_config, X, y, epochs=epochs)
end = time.time()
elapsed = end - start
print(f"Training time: {int(elapsed // 60)} min {int(elapsed % 60)} sec")

model_gru.summary()


Epoch 1/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.0485 - loss: 7.1782
Epoch 2/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.0600 - loss: 6.6007
Epoch 3/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0778 - loss: 6.2932
Epoch 4/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.0912 - loss: 6.0663
Epoch 5/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 24s 30ms/step - accuracy: 0.1000 - loss: 5.8871
Epoch 6/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.1070 - loss: 5.7263
Epoch 7/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.1160 - loss: 5.5720
Epoch 8/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.1263 - loss: 5.4220
Epoch 9/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.1376 - loss: 5.2789
Epoch 10/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.1478 - loss: 5.1435
Epoch 11/50
809/809 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.1579 - loss: 5.0161
Epoch 12/50
809/809 ━━━━━━━━━━

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 83, 32)         │       159,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4984)           │       164,472 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 990,890 (3.78 MB)

 Trainable params: 330,296 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 660,594 (2.52 MB)

In [20]:
my_sentence = "the castle"

predict_next_n_words(model_gru, tokenizer, my_sentence, 5)

the castle and the land of the land
